# Action-Level Recommendation Explanations

This notebook explains the **simulated optimizer schedule** at the action level.

The explanations are based on:

- the optimized compressor states,
- predicted receiver pressure,
- reserve margin,
- the frozen baseline interval energy,
- the optimizer's declared safety constraints.

They are **constraint- and schedule-based explanations**.

They are not SHAP explanations and they do not establish physical causality. The interval energy comparison is not interpreted as the causal effect of one individual compressor action.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd().resolve()

if (
    not (ROOT / "ml").exists()
    and (ROOT.parent / "ml").exists()
):
    ROOT = ROOT.parent

if not (ROOT / "ml").exists():
    raise RuntimeError(
        "Could not locate repository root."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(ROOT),
    )

from ml.control.baseline import (
    BaselineControllerConfig,
    simulate_baseline,
)
from ml.explainability.actions import (
    explain_schedule,
    explanations_frame,
)
from ml.optimization.scheduler import (
    OptimizationConfig,
    optimize_schedule,
)
from ml.twin.compressor import CompressorSpec
from ml.twin.physics import TwinParameters

print("Repository root:", ROOT)

Repository root: /mnt/c/Users/Dell ProMax Tower T2/Downloads/code/xAI-Compressor


## Load the same frozen simulation assumptions

This notebook does not introduce a new optimization scenario. It recreates the same one-hour nominal scenario used in notebook 07.

In [2]:
with (
    ROOT
    / "configs"
    / "compressors.yaml"
).open(
    "r",
    encoding="utf-8",
) as handle:
    compressor_config = yaml.safe_load(handle)

with (
    ROOT
    / "configs"
    / "twin.yaml"
).open(
    "r",
    encoding="utf-8",
) as handle:
    twin_config = yaml.safe_load(handle)

with (
    ROOT
    / "configs"
    / "optimization.yaml"
).open(
    "r",
    encoding="utf-8",
) as handle:
    optimization_raw = yaml.safe_load(
        handle
    )["optimization"]

compressors = [
    CompressorSpec(
        id=str(item["id"]),
        kind=str(item["kind"]),
        max_mass_flow_kg_s=float(
            item["max_mass_flow_kg_s"]
        ),
        rated_power_kw=float(
            item["rated_power_kw"]
        ),
        idle_power_kw=float(
            item["idle_power_kw"]
        ),
        min_load_fraction=float(
            item["min_load_fraction"]
        ),
        min_on_seconds=float(
            item["min_on_seconds"]
        ),
        min_off_seconds=float(
            item["min_off_seconds"]
        ),
    )
    for item in compressor_config[
        "compressors"
    ]
]

pressure_config = compressor_config[
    "pressure"
]

controller_raw = compressor_config[
    "controller"
]

baseline_controller = (
    BaselineControllerConfig(
        target_bar_g=float(
            pressure_config[
                "target_bar_g"
            ]
        ),
        lower_band_bar_g=float(
            pressure_config[
                "lower_band_bar_g"
            ]
        ),
        upper_band_bar_g=float(
            pressure_config[
                "upper_band_bar_g"
            ]
        ),
        safety_min_bar_g=float(
            pressure_config[
                "safety_min_bar_g"
            ]
        ),
        safety_max_bar_g=float(
            pressure_config[
                "safety_max_bar_g"
            ]
        ),
        pressure_gain_kg_s_per_bar=float(
            controller_raw[
                "pressure_gain_kg_s_per_bar"
            ]
        ),
    )
)

air_config = twin_config["air"]
receiver_config = twin_config["receiver"]

parameters = TwinParameters(
    volume_m3=float(
        receiver_config["volume_m3"]
    ),
    temperature_k=float(
        air_config["temperature_k"]
    ),
    gas_constant_j_per_kg_k=float(
        air_config[
            "gas_constant_j_per_kg_k"
        ]
    ),
    ambient_pressure_pa=float(
        air_config[
            "ambient_pressure_pa"
        ]
    ),
)

solver_raw = optimization_raw[
    "solver"
]

optimizer_config = OptimizationConfig(
    interval_seconds=float(
        optimization_raw[
            "interval_seconds"
        ]
    ),
    reserve_mass_flow_kg_s=float(
        optimization_raw[
            "reserve_mass_flow_kg_s"
        ]
    ),
    startup_penalty_kwh=float(
        optimization_raw[
            "startup_penalty_kwh"
        ]
    ),
    overpressure_penalty_kwh_per_bar_hour=float(
        optimization_raw[
            "overpressure_penalty_kwh_per_bar_hour"
        ]
    ),
    terminal_pressure_min_bar_g=float(
        optimization_raw[
            "terminal_pressure_min_bar_g"
        ]
    ),
    time_limit_seconds=float(
        solver_raw[
            "time_limit_seconds"
        ]
    ),
    mip_rel_gap=float(
        solver_raw[
            "mip_rel_gap"
        ]
    ),
)

nominal_leak_kg_s = float(
    compressor_config[
        "scenario"
    ][
        "nominal_leak_kg_s"
    ]
)

## Recreate baseline and optimized schedules

In [3]:
block_seconds = 15 * 60

baseline_demand = (
    [0.070] * block_seconds
    + [0.110] * block_seconds
    + [0.145] * block_seconds
    + [0.090] * block_seconds
)

interval_seconds = int(
    optimizer_config.interval_seconds
)

intervals_per_block = (
    block_seconds
    // interval_seconds
)

optimizer_demand = (
    [0.070] * intervals_per_block
    + [0.110] * intervals_per_block
    + [0.145] * intervals_per_block
    + [0.090] * intervals_per_block
)

baseline = simulate_baseline(
    baseline_demand,
    leak_mass_flow_kg_s=(
        nominal_leak_kg_s
    ),
    initial_pressure_bar_g=7.0,
    parameters=parameters,
    compressors=compressors,
    controller_config=(
        baseline_controller
    ),
    timestep_seconds=1.0,
)

optimized = optimize_schedule(
    optimizer_demand,
    leak_mass_flow_kg_s=(
        nominal_leak_kg_s
    ),
    initial_pressure_bar_g=7.0,
    parameters=parameters,
    compressors=compressors,
    target_bar_g=float(
        pressure_config[
            "target_bar_g"
        ]
    ),
    safety_min_bar_g=float(
        pressure_config[
            "safety_min_bar_g"
        ]
    ),
    safety_max_bar_g=float(
        pressure_config[
            "safety_max_bar_g"
        ]
    ),
    config=optimizer_config,
)

schedule = optimized.schedule

print(
    "Baseline energy:",
    baseline[
        "cumulative_energy_kwh"
    ].iloc[-1],
)

print(
    "Optimized energy:",
    optimized.energy_kwh,
)

Baseline energy: 25.311174554824547
Optimized energy: 25.217708333326183


## Aggregate the 1-second baseline into the optimizer's 60-second intervals

This gives an interval-by-interval electrical-energy comparison without claiming that the difference is caused by any one individual device action.

In [4]:
baseline = baseline.copy()

baseline[
    "optimizer_interval"
] = (
    (
        baseline[
            "time_seconds"
        ] - 1.0
    )
    // interval_seconds
).astype(int)

baseline_interval_energy = (
    baseline.groupby(
        "optimizer_interval"
    )[
        "energy_kwh"
    ]
    .sum()
    .reindex(
        range(len(schedule))
    )
)

if baseline_interval_energy.isna().any():
    raise RuntimeError(
        "Baseline interval aggregation "
        "is incomplete."
    )

baseline_interval_energy.head()

optimizer_interval
0    0.298958
1    0.298958
2    0.298958
3    0.298958
4    0.298958
Name: energy_kwh, dtype: float64

## Generate structured action explanations

In [5]:
explanations = explain_schedule(
    schedule,
    target_bar_g=float(
        pressure_config[
            "target_bar_g"
        ]
    ),
    safety_min_bar_g=float(
        pressure_config[
            "safety_min_bar_g"
        ]
    ),
    safety_max_bar_g=float(
        pressure_config[
            "safety_max_bar_g"
        ]
    ),
    required_reserve_kg_s=(
        optimizer_config
        .reserve_mass_flow_kg_s
    ),
    baseline_interval_energy_kwh=(
        baseline_interval_energy
        .tolist()
    ),
)

explanation_frame = (
    explanations_frame(
        explanations
    )
)

explanation_frame[
    [
        "interval_index",
        "time_seconds",
        "recommended_action",
        "pressure_start_bar_g",
        "pressure_end_bar_g",
        "safety_margin_bar",
        "reserve_available_kg_s",
        "interval_energy_delta_kwh",
        "binding_constraints",
    ]
].head(10)

,interval_index,time_seconds,recommended_action,pressure_start_bar_g,pressure_end_bar_g,safety_margin_bar,reserve_available_kg_s,interval_energy_delta_kwh,binding_constraints
0,0,60.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.000000,7.005135,0.505135,0.065,0.007708,()
1,1,120.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.005135,7.010270,0.510270,0.065,0.007708,()
2,2,180.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.010270,7.015405,0.515405,0.065,0.007708,()
3,3,240.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.015405,7.020540,0.520540,0.065,0.007708,()
4,4,300.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.020540,7.025675,0.525675,0.065,0.007708,()
5,5,360.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.025675,7.030810,0.530810,0.065,0.007708,()
6,6,420.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.030810,7.035945,0.535945,0.065,0.007708,()
7,7,480.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.035945,7.041080,0.541080,0.065,0.007708,()
8,8,540.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.041080,7.046215,0.546215,0.065,0.007708,()
9,9,600.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,7.046215,7.051350,0.551350,0.065,0.007708,()


## Inspect the most informative decision points

The notebook surfaces:

- the lowest-pressure interval,
- the highest-pressure interval,
- the interval with the largest energy reduction versus baseline,
- intervals where fixed-compressor state changes occur.

These are explanation examples, not a cherry-picked performance metric.

In [6]:
decision_indices: set[int] = {
    int(
        schedule[
            "pressure_bar_g"
        ].idxmin()
    ),
    int(
        schedule[
            "pressure_bar_g"
        ].idxmax()
    ),
    int(
        explanation_frame[
            "interval_energy_delta_kwh"
        ].idxmin()
    ),
}

for column in [
    item
    for item in schedule.columns
    if (
        item.startswith("fixed_")
        and item.endswith("_on")
    )
]:
    transitions = (
        schedule[column]
        != schedule[column].shift(
            fill_value=False
        )
    )

    decision_indices.update(
        int(index)
        for index
        in schedule.index[
            transitions
        ].tolist()
    )

decision_points = (
    explanation_frame.loc[
        sorted(decision_indices)
    ]
    .copy()
)

decision_points[
    [
        "interval_index",
        "time_seconds",
        "recommended_action",
        "reason",
        "binding_constraints",
    ]
]

,interval_index,time_seconds,recommended_action,reason,binding_constraints
0,0,60.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,"Receiver pressure begins at target, so the sch...",()
18,18,1140.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,"Receiver pressure begins below target, so the ...","(pressure_safety_min,)"
30,30,1860.0,fixed_1 ON; fixed_2 ON; vsd_1 37.5%,"Receiver pressure begins below target, so the ...","(pressure_safety_min,)"
51,51,3120.0,fixed_1 ON; fixed_2 ON; vsd_1 OFF,"Receiver pressure begins above target, so the ...",()
52,52,3180.0,fixed_1 ON; fixed_2 OFF; vsd_1 20.0%,"Receiver pressure begins above target, so the ...",()


## Explanation validity gate

Before exporting these explanations, verify that every explanation is marked simulated and non-causal, and that the explanation count matches the optimized horizon.

In [7]:
explanation_acceptance = {
    "count_matches_schedule": (
        len(explanations)
        == len(schedule)
    ),
    "all_simulated": all(
        item.evidence_class
        == "simulated"
        for item in explanations
    ),
    "no_causal_claims": all(
        not item.causal_claim
        for item in explanations
    ),
    "baseline_comparison_complete": bool(
        explanation_frame[
            "baseline_interval_energy_kwh"
        ].notna().all()
    ),
}

explanation_acceptance[
    "all_passed"
] = all(
    explanation_acceptance.values()
)

pd.Series(
    explanation_acceptance,
    name="passed",
)

count_matches_schedule          True
all_simulated                   True
no_causal_claims                True
baseline_comparison_complete    True
all_passed                      True
Name: passed, dtype: bool

In [8]:
if not explanation_acceptance[
    "all_passed"
]:
    raise RuntimeError(
        "Action explanation acceptance "
        "gate failed."
    )

print(
    "Action explanation acceptance gate: PASS"
)

Action explanation acceptance gate: PASS


## Write the action-explanation evidence report

In [9]:
REPORT_PATH = (
    ROOT
    / "docs"
    / "action_explanations.json"
)

report = {
    "evidence_class": "simulated",
    "explanation_basis": (
        "optimizer schedule, pressure "
        "constraints, reserve margin, "
        "and interval energy comparison"
    ),
    "causal_claim": False,
    "acceptance": (
        explanation_acceptance
    ),
    "summary": {
        "interval_count":
            len(explanations),
        "decision_point_count":
            len(decision_points),
        "minimum_safety_margin_bar":
            float(
                explanation_frame[
                    "safety_margin_bar"
                ].min()
            ),
        "binding_constraint_intervals":
            int(
                explanation_frame[
                    "binding_constraints"
                ].map(bool).sum()
            ),
    },
    "decision_points": (
        decision_points.to_dict(
            orient="records"
        )
    ),
    "limitations": [
        (
            "Explanations describe the "
            "simulated optimizer schedule."
        ),
        (
            "Interval energy differences "
            "versus baseline are not the "
            "causal effect of a single "
            "compressor action."
        ),
        (
            "These explanations do not "
            "establish physical root cause."
        ),
        (
            "Robustness to demand and model "
            "uncertainty is evaluated "
            "separately."
        ),
    ],
}

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Action explanation report written to:",
    REPORT_PATH,
)

Action explanation report written to: /mnt/c/Users/Dell ProMax Tower T2/Downloads/code/xAI-Compressor/docs/action_explanations.json


## Interpretation

This layer explains **why a simulated schedule is feasible and how it differs from the conventional baseline**.

It should not be presented as proof that one compressor action caused a particular amount of energy saving.

The next methodological step is robustness/sensitivity analysis: apply the frozen nominal recommendation under plausible demand, leak, and model perturbations and check whether pressure safety remains acceptable.